In [1]:
import pandas as pd
import ast
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import defaultdict
from itertools import combinations



#  LOAD DATA

patient_data = pd.read_csv('patient_records.csv')
disease_data = pd.read_csv('Diseases_Symptoms.csv')
pattern      = pd.read_csv('pattern.csv')

patient_data = patient_data.head(1000)
def parse_conditions(val):
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else [str(val).strip()]
    except Exception:
        if isinstance(val, str) and val.strip() not in ('', 'nan'):
            return [v.strip() for v in val.split(',')]
        return []

patient_data['_conditions'] = patient_data['Medical Condition'].apply(parse_conditions)
patient_data['PatientID']   = patient_data['PatientID'].str.strip().str.upper()



#  BUILD CONDITION-PAIR FREQUENCY TABLE
#  For every pair of conditions that appear
#  together, count what disease those patients
#  developed next. Built once at startup.

NON_DISEASE_OUTCOMES = {
    'Condition managed — monitoring only',
    'Palliative care recommended',
    'End of disease trajectory',
    'Stable — no progression expected',
    'Remission achieved',
    'End of trajectory',
}

pair_next_counts = defaultdict(lambda: defaultdict(float))

for _, record in patient_data.iterrows():
    conds  = record['_conditions']
    next_d = str(record.get('Predicted_Next_Disease', '')).strip()
    if not next_d or next_d in ('nan', '') or next_d in NON_DISEASE_OUTCOMES:
        continue
    for pair in combinations(sorted(conds), 2):
        pair_next_counts[pair][next_d] += 1.0


#  BUILD PROGRESSION CHAIN STRUCTURES
#  Each chain is a known disease sequence
#  e.g. Obesity → Diabetes → Hypertension → Stroke

chain_sequences = {
    name: grp.sort_values('Step_Number')['Disease_At_This_Step'].tolist()
    for name, grp in pattern.groupby('Chain_Name')
}

chain_lookup = (
    pattern[[
        'Disease_At_This_Step', 'Chain_Name', 'Step_Number',
        'Total_Steps_In_Chain', 'Avg_Years_To_Next_Disease',
        'Next_Disease_In_Progression', 'Key_Risk_Factors', 'Is_Last_Disease'
    ]]
    .drop_duplicates(subset='Disease_At_This_Step')
    .set_index('Disease_At_This_Step')
)


#  STEP 1 — PROGRESSION CHAIN MATCHING
#  Check if the patient fits a known disease
#  progression chain. They must share at least
#  2 conditions with the chain to qualify.

def find_patient_chain(conditions_list):
    if len(conditions_list) < 2:
        return None

    cond_set   = set(conditions_list)
    best_chain = None
    best_score = (-1, -1)

    for chain_name, sequence in chain_sequences.items():
        overlap = len(cond_set & set(sequence))
        if overlap < 2:
            continue

        furthest_step    = -1
        furthest_disease = None
        for disease in conditions_list:
            if disease not in chain_lookup.index:
                continue
            entry = chain_lookup.loc[disease]
            if entry['Chain_Name'] != chain_name:
                continue
            step = int(entry['Step_Number'])
            if step > furthest_step:
                furthest_step    = step
                furthest_disease = disease

        if furthest_disease is None:
            continue

        score = (overlap, furthest_step)
        if score > best_score:
            best_score   = score
            best_disease = furthest_disease
            best_chain   = chain_name

    if best_chain is None:
        return None

    entry = chain_lookup.loc[best_disease]
    try:
        avg_years = float(entry['Avg_Years_To_Next_Disease'])
    except Exception:
        avg_years = 0.0

    next_disease = str(entry['Next_Disease_In_Progression']).strip()
    is_end = (
        str(entry['Is_Last_Disease']).strip().lower() == 'yes'
        or next_disease in ('', 'nan', 'End of trajectory')
    )

    return {
        'chain_name'   : entry['Chain_Name'],
        'current'      : best_disease,
        'step'         : int(entry['Step_Number']),
        'total_steps'  : int(entry['Total_Steps_In_Chain']),
        'next_disease' : next_disease if not is_end else None,
        'years_to_next': avg_years,
        'is_end'       : is_end,
        'overlap'      : best_score[0],
        'risk_factors' : str(entry.get('Key_Risk_Factors', '')),
    }


#  STEP 2 — SIMILAR PATIENT LOOKUP
#  Find patients in the database who share
#  the same conditions, age range, and BMI.
#  Learn from what they developed next.

def find_similar_patients(conditions, age=None, bmi=None):
    cond_set = set(conditions)
    strong   = {}
    moderate = {}

    for _, record in patient_data.iterrows():
        other_conds = set(record['_conditions'])
        overlap     = len(cond_set & other_conds)
        if overlap == 0:
            continue

        next_d = str(record.get('Predicted_Next_Disease', '')).strip()
        if not next_d or next_d in NON_DISEASE_OUTCOMES or next_d in ('nan', '', 'End of trajectory'):
            continue

        union   = len(cond_set | other_conds)
        jaccard = overlap / union if union else 0

        similarity_bonus = 0.0
        try:
            rec_age = float(record.get('Age', 0) or 0)
            rec_bmi = float(record.get('BMI', 0) or 0)
            if age and rec_age:
                similarity_bonus += 0.15 * max(0, 1 - abs(age - rec_age) / 40)
            if bmi and rec_bmi:
                similarity_bonus += 0.15 * max(0, 1 - abs(bmi - rec_bmi) / 20)
        except Exception:
            pass

        score = jaccard + similarity_bonus

        if overlap >= 2:
            strong[next_d]   = strong.get(next_d, 0.0) + score
        else:
            moderate[next_d] = moderate.get(next_d, 0.0) + score * 0.5

    combined = dict(strong)
    for disease, score in moderate.items():
        if disease not in combined:
            combined[disease] = score
        else:
            combined[disease] += score * 0.3

    # Amplify using the condition-pair frequency table
    pair_votes = defaultdict(float)
    for pair in combinations(sorted(cond_set), 2):
        if pair in pair_next_counts:
            for disease, count in pair_next_counts[pair].items():
                pair_votes[disease] += count

    if pair_votes:
        max_votes = max(pair_votes.values())
        for disease, votes in pair_votes.items():
            combined[disease] = combined.get(disease, 0.0) + (votes / max_votes) * 1.5

    # Fallback: look at patients in the same age and BMI range
    if age and bmi and (not combined or max(combined.values(), default=0) < 0.5):
        for _, record in patient_data.iterrows():
            try:
                rec_age = float(record.get('Age', 0) or 0)
                rec_bmi = float(record.get('BMI', 0) or 0)
            except Exception:
                continue
            if abs(rec_age - age) > 10 or abs(rec_bmi - bmi) > 5:
                continue
            next_d = str(record.get('Predicted_Next_Disease', '')).strip()
            if not next_d or next_d in NON_DISEASE_OUTCOMES or next_d in ('nan', '', 'End of trajectory'):
                continue
            combined[next_d] = combined.get(next_d, 0.0) + 0.1

    return combined



#  STEP 3 — CLINICAL RULES
#  Medical evidence showing which conditions
#  commonly lead to which diseases, with
#  additional risk boosts based on age and BMI.

CLINICAL_RULES = {
    'Chronic Kidney Disease'    : ['Kidney Failure', 'Anemia due to Chronic Kidney Disease', 'Hypertension'],
    'Diabetic Kidney Disease'   : ['Chronic Kidney Disease', 'Kidney Failure'],
    'Obesity'                   : ['Type 2 Diabetes', 'Hypertension', 'Hypertensive Heart Disease'],
    'Type 2 Diabetes'           : ['Diabetic Kidney Disease', 'Diabetic retinopathy', 'Neuropathic Pain'],
    'Hypertension'              : ['Hypertensive Heart Disease', 'Stroke', 'Heart Block'],
    'Hypertensive Heart Disease': ['Heart Block', 'Atrial Fibrillation'],
    'Atrial Fibrillation'       : ['Stroke', 'Pulmonary Congestion'],
    'Depression'                : ['Chronic Fatigue Syndrome', 'Primary Insomnia'],
    'Osteoporosis'              : ['Fracture', 'Fracture of the Shoulder'],
    'Rheumatoid Arthritis'      : ['Osteoporosis', 'Arthritis'],
    'Lupus'                     : ['Chronic Kidney Disease', 'Anemia of Chronic Disease'],
    'Asthma'                    : ['Emphysema', 'Atelectasis'],
    'Cirrhosis'                 : ['Esophageal Varices', 'Liver Cancer', 'Hepatic Encephalopathy'],
    'Psoriasis'                 : ['Arthritis', 'Lymphoma'],
    'Fibromyalgia'              : ['Chronic Fatigue Syndrome', 'Primary Insomnia', 'Neuralgia'],
}

def apply_clinical_rules(conditions, age, bmi):
    cond_set = set(conditions)
    scores   = {}

    for condition, likely_next in CLINICAL_RULES.items():
        if condition in cond_set:
            for disease in likely_next:
                if disease not in cond_set:
                    scores[disease] = scores.get(disease, 0.0) + 10.0

    if age and age > 65:
        for disease in ['Stroke', 'Atrial Fibrillation', 'Osteoporosis', 'Macular Degeneration']:
            scores[disease] = scores.get(disease, 0.0) + 8.0
    elif age and age > 50:
        for disease in ['Type 2 Diabetes', 'Hypertension', 'Diabetic retinopathy']:
            scores[disease] = scores.get(disease, 0.0) + 4.0

    if bmi and bmi >= 35:
        for disease in ['Type 2 Diabetes', 'Hypertensive Heart Disease', 'Heart Block']:
            scores[disease] = scores.get(disease, 0.0) + 7.0
    elif bmi and bmi >= 30:
        for disease in ['Type 2 Diabetes', 'Hypertension']:
            scores[disease] = scores.get(disease, 0.0) + 4.0

    return scores


#  COMBINE ALL SIGNALS AND RANK

def scale_to_one(scores):
    if not scores:
        return {}
    lo, hi = min(scores.values()), max(scores.values())
    if hi == lo:
        return {k: 1.0 for k in scores}
    return {k: (v - lo) / (hi - lo) for k, v in scores.items()}


def rank_predictions(chain_match, similar_patient_scores, rule_scores, current_conditions):
    chain_scores = {}
    if chain_match and chain_match['next_disease']:
        chain_scores[chain_match['next_disease']] = 1.0

    chain_scaled  = scale_to_one(chain_scores)
    cohort_scaled = scale_to_one(similar_patient_scores)
    rules_scaled  = scale_to_one(rule_scores)

    if chain_match and chain_match['next_disease']:
        w_chain, w_cohort, w_rules = 0.50, 0.35, 0.15
    else:
        w_chain, w_cohort, w_rules = 0.00, 0.70, 0.30

    all_candidates = set(chain_scaled) | set(cohort_scaled) | set(rules_scaled)
    final_scores   = {}

    for disease in all_candidates:
        if disease in current_conditions:
            continue
        score = (
            w_chain  * chain_scaled.get(disease, 0)
            + w_cohort * cohort_scaled.get(disease, 0)
            + w_rules  * rules_scaled.get(disease, 0)
        )
        if score > 0:
            final_scores[disease] = score

    if not final_scores:
        return []

    top5          = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    diseases      = [d for d, _ in top5]
    values        = np.array([v for _, v in top5])
    exp_vals      = np.exp((values - values.max()) / 0.15)
    probabilities = exp_vals / exp_vals.sum()

    return sorted(zip(diseases, probabilities), key=lambda x: x[1], reverse=True)



#  RUN PREDICTION FOR AN EXISTING PATIENT

def predict_patient(record):
    conditions = record['_conditions']
    age        = record.get('Age')
    bmi        = record.get('BMI')

    W   = 58
    SEP = "─" * W

    if len(conditions) < 3:
        print(f"\n  {SEP}")
        print(f"  {record['Name'].title()} currently has {len(conditions)} condition(s) on record.")
        print(f"  At least 3 conditions are needed before a prediction can be made.")
        print(f"  {SEP}\n")
        return

    print(f"\n  {SEP}")
    print(f"  Patient    : {record['Name'].title()}   (ID: {record['PatientID']})")
    print(f"  Age / BMI  : {age} / {bmi}")
    print(f"  Conditions : {', '.join(conditions)}")
    print(f"  {SEP}")

    chain_match = find_patient_chain(conditions)

    if chain_match:
        if chain_match['is_end']:
            progression_label = "end of known progression"
        else:
            progression_label = f"{chain_match['next_disease']}  (expected in ~{chain_match['years_to_next']:.1f} years)"
        print(f"  Progression Path : {chain_match['chain_name']}")
        print(f"  Current Stage    : Step {chain_match['step']} of {chain_match['total_steps']}  →  {progression_label}")
        if chain_match['risk_factors'] not in ('', 'nan', 'None'):
            print(f"  Key Risk Factors : {chain_match['risk_factors']}")
    else:
        print(f"  Progression Path : No matching chain found for this condition combination.")
        print(f"  Prediction will be based on similar patients and clinical evidence.")

    print(f"  {SEP}")

    similar_scores = find_similar_patients(conditions, age, bmi)
    rule_scores    = apply_clinical_rules(conditions, age, bmi)
    predictions    = rank_predictions(chain_match, similar_scores, rule_scores, set(conditions))

    if not predictions:
        print(f"  Not enough data to make a reliable prediction at this time.\n")
        return

    print(f"  PREDICTED NEXT CONDITIONS")
    print(f"  Based on {len(similar_scores)} similar patient records in the database.")
    print(f"  {SEP}")

    for rank, (disease, probability) in enumerate(predictions[:3], 1):
        bar = '█' * max(1, int(probability * 28))
        tag = "  ← highest risk" if rank == 1 else ""
        print(f"  {rank}. {disease:<36} {probability*100:5.1f}%  {bar}{tag}")

    top_disease = predictions[0][0]
    reasons     = []
    if chain_match and chain_match.get('next_disease') == top_disease:
        reasons.append(f"follows the known progression: {chain_match['chain_name']}")
    if top_disease in similar_scores:
        reasons.append(f"commonly developed by patients with the same conditions")
    for condition, next_diseases in CLINICAL_RULES.items():
        if condition in set(conditions) and top_disease in next_diseases:
            reasons.append(f"medical evidence links {condition} to {top_disease}")
            break

    if reasons:
        print(f"\n  Reason  : {' | '.join(reasons)}")

    print(f"  {SEP}")
    print(f"  This is a clinical support tool. Please consult a doctor for diagnosis.\n")



#  REGISTER A NEW PATIENT

def register_new_patient():
    W   = 58
    SEP = "─" * W

    print(f"\n  {SEP}")
    print(f"  NEW PATIENT REGISTRATION")
    print(f"  {SEP}")

    pid  = input("  Patient ID  : ").strip().upper()
    name = input("  Full Name   : ").strip()
    try:
        age = float(input("  Age         : ").strip())
    except ValueError:
        age = None
    try:
        bmi = float(input("  BMI         : ").strip())
    except ValueError:
        bmi = None

    print(f"\n  Enter the patient's symptoms one at a time.")
    print(f"  Type  DONE  when finished.\n")

    symptoms = []
    while True:
        entry = input("  Symptom     : ").strip()
        if entry.lower() == 'done':
            break
        if entry:
            symptoms.append(entry.lower())

    if not symptoms:
        print(f"\n  No symptoms were entered. Registration cancelled.\n")
        return

    name_col = 'Disease Name' if 'Disease Name' in disease_data.columns else disease_data.columns[0]
    entered  = set(symptoms)
    matches  = []

    for _, row in disease_data.iterrows():
        disease_name = str(row[name_col]).strip()
        raw_symptoms = str(row.get('Symptoms', '')).lower()
        known_syms   = set(s.strip() for s in raw_symptoms.split(',') if s.strip() not in ('', 'nan'))
        overlap      = sum(1 for e in entered if any(e in s or s in e for s in known_syms))
        if overlap > 0:
            matches.append((disease_name, overlap, len(known_syms)))

    matches.sort(key=lambda x: x[1], reverse=True)

    print(f"\n  {SEP}")
    print(f"  POSSIBLE CONDITIONS FOR: {name.upper()}")
    print(f"  {SEP}")

    if not matches:
        print(f"  No matching conditions could be found based on the symptoms entered.")
        print(f"  Please see a doctor for a proper evaluation.")
    else:
        print(f"  {'Condition':<38} {'Symptoms Matched'}")
        print(f"  {'─' * W}")
        top_matches = matches[:5]
        for disease, matched, total in top_matches:
            dots = '●' * matched
            print(f"  {disease:<38} {matched} of {total}   {dots}")

        top_condition = top_matches[0][0]
        new_entry = {
            'PatientID'              : pid,
            'Name'                   : name,
            'Age'                    : age,
            'BMI'                    : bmi,
            'Medical Condition'      : f'["{top_condition}"]',
            'Predicted_Next_Disease' : '',
        }

        global patient_data
        patient_data = pd.concat(
            [patient_data, pd.DataFrame([new_entry])], ignore_index=True
        )
        patient_data['_conditions'] = patient_data['Medical Condition'].apply(parse_conditions)

        print(f"\n  Patient registered successfully.")
        print(f"  Most likely condition recorded : {top_condition}")

    print(f"\n  Please visit a doctor for a confirmed diagnosis.")
    print(f"  {SEP}\n")


#  MODEL EVALUATION METRICS
#  Computed once at startup using patients who
#  have a recorded Predicted_Next_Disease.
#  Measures how well the model would have
#  predicted each patient's actual outcome.

def compute_model_metrics(sample_size=300):
    """
    Evaluates model accuracy on a random sample of patients
    who have at least 3 conditions and a known next disease.
    Fast: runs in seconds instead of minutes.
    """
    eligible = patient_data[
        patient_data['_conditions'].apply(len) >= 3
    ].copy()

    eligible = eligible[
        eligible['Predicted_Next_Disease'].notna() &
        (~eligible['Predicted_Next_Disease'].astype(str).str.strip().isin(
            NON_DISEASE_OUTCOMES | {'', 'nan', 'End of trajectory'}
        ))
    ]

    total = len(eligible)
    if total == 0:
        return None

    # Sample for speed — statistically representative at 300 patients
    sample = eligible.sample(n=min(sample_size, total), random_state=42)

    correct   = 0
    top3_hit  = 0
    predicted = 0

    for _, record in sample.iterrows():
        actual     = str(record['Predicted_Next_Disease']).strip()
        conditions = record['_conditions']
        age        = record.get('Age')
        bmi        = record.get('BMI')

        chain_match    = find_patient_chain(conditions)
        similar_scores = find_similar_patients(conditions, age, bmi)
        rule_scores    = apply_clinical_rules(conditions, age, bmi)
        predictions    = rank_predictions(chain_match, similar_scores, rule_scores, set(conditions))

        if not predictions:
            continue

        predicted += 1
        top_names  = [d for d, _ in predictions[:3]]

        if top_names[0] == actual:
            correct  += 1
            top3_hit += 1
        elif actual in top_names:
            top3_hit += 1

    accuracy      = (correct  / predicted * 100) if predicted else 0.0
    top3_accuracy = (top3_hit / predicted * 100) if predicted else 0.0
    coverage      = (predicted / len(sample) * 100) if len(sample) else 0.0

    return {
        'accuracy'        : accuracy,
        'top3_accuracy'   : top3_accuracy,
        'coverage'        : coverage,
        'total_eligible'  : total,
        'sample_size'     : len(sample),
        'predicted'       : predicted,
        'correct'         : correct,
        'top3_hit'        : top3_hit,
    }


def show_model_metrics(m=None):
    W   = 58
    SEP = "─" * W

    if m is None:
        print(f"\n  {SEP}")
        print(f"  Evaluating model on a sample of patient records...")
        m = compute_model_metrics()

    if m is None:
        print(f"  Not enough labelled data to evaluate the model.\n")
        return

    print(f"\n  {SEP}")
    print(f"  {'MODEL PERFORMANCE REPORT':^{W}}")
    print(f"  {SEP}")
    print(f"  Eligible patients in database  : {m['total_eligible']:,}")
    print(f"  Sample evaluated               : {m['sample_size']:,}")
    print(f"  Predictions generated          : {m['predicted']:,}")
    print(f"  {SEP}")

    bar1 = '█' * int(m['accuracy']      / 100 * 30)
    bar3 = '█' * int(m['top3_accuracy'] / 100 * 30)
    barC = '█' * int(m['coverage']      / 100 * 30)

    print(f"  Top-1 Accuracy   {m['accuracy']:5.1f}%  {bar1}")
    print(f"  Top-3 Accuracy   {m['top3_accuracy']:5.1f}%  {bar3}")
    print(f"  Coverage         {m['coverage']:5.1f}%  {barC}")
    print(f"  {SEP}")
    print(f"  Exact matches    : {m['correct']:,} of {m['predicted']:,} patients")
    print(f"  In top 3         : {m['top3_hit']:,} of {m['predicted']:,} patients")
    print(f"  {SEP}")

    if m['accuracy'] >= 70:
        assessment = "Strong — the model predicts correctly most of the time."
    elif m['accuracy'] >= 50:
        assessment = "Moderate — the model is right more often than not."
    elif m['accuracy'] >= 30:
        assessment = "Fair — useful as a support tool alongside clinical judgment."
    else:
        assessment = "Developing — more labelled patient data will improve accuracy."

    print(f"  Assessment : {assessment}")
    print(f"  {SEP}\n")



def main():
    W   = 58
    SEP = "-" * W

    print(f"\n  {SEP}")
    print(f"  {'DISEASE PROGRESSION PREDICTION SYSTEM':^{W}}")
    print(f"  {SEP}")
    print(f"  Patient records loaded : {len(patient_data):,}")
    print(f"  Progression chains     : {len(chain_sequences)}")
    print(f"  {SEP}")
    print(f"  Computing model accuracy on a sample of records...")

    _metrics = compute_model_metrics()

    if _metrics:
        print(f"  Top-1 Accuracy   : {_metrics['accuracy']:.1f}%   "
              f"(exact match on first prediction)")
        print(f"  Top-3 Accuracy   : {_metrics['top3_accuracy']:.1f}%   "
              f"(correct answer in top 3)")
        print(f"  Coverage         : {_metrics['coverage']:.1f}%   "
              f"(sample: {_metrics['sample_size']:,} patients)")

    print(f"  {SEP}\n")

    while True:
        choice = input("  [E] Existing Patient   [N] New Patient   [M] Model Performance   [Q] Quit\n  > ").strip().lower()

        if choice in ('q', 'quit', 'exit'):
            print(f"\n  Session ended. Goodbye.\n")
            break
        elif choice == 'n':
            register_new_patient()
        elif choice == 'm':
            show_model_metrics(_metrics)
        elif choice == 'e':
            pid   = input("\n  Enter Patient ID : ").strip().upper()
            found = patient_data[patient_data['PatientID'] == pid]
            if found.empty:
                print(f"\n  No patient found with ID '{pid}'. Please check and try again.\n")
            else:
                predict_patient(found.iloc[0])
        else:
            print(f"\n  Please type E, N, or Q.\n")


if __name__ == '__main__':
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'patient_records.csv'